In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!nvidia-smi

Wed Apr 29 09:52:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!git clone https://github.com/RajeevTricel/gemma-seo-agent.git
%cd gemma-seo-agent

fatal: destination path 'gemma-seo-agent' already exists and is not an empty directory.
/kaggle/working/gemma-seo-agent/gemma-seo-agent


In [ ]:
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes huggingface_hub sentencepiece protobuf --upgrade-strategy only-if-needed

In [4]:
import torch
import transformers
import datasets
import peft
import trl
import bitsandbytes
import huggingface_hub

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)
print("All imports working")

Torch: 2.10.0+cu128
CUDA available: True
Transformers: 5.7.0
All imports working


In [5]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

print("TRL trainer imports working")

TRL trainer imports working


In [6]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="data/train.jsonl", split="train")

print(dataset)
print(dataset[0])

Dataset({
    features: ['messages', 'metadata'],
    num_rows: 400
})
{'messages': [{'role': 'system', 'content': 'You are an evidence-led SEO and marketing assistant. Diagnose using provided data, avoid guessing, prioritise by business impact, and return practical actions using the required format: Diagnosis, Evidence, Priority, Fix, Next action.'}, {'role': 'user', 'content': 'Here is GSC data: page /industrial-tanks clicks -42%, impressions -8%, avg position 4.1 to 8.9, CTR 5.2% to 3.1%. What matters?'}, {'role': 'assistant', 'content': 'Diagnosis: This is mainly a ranking-loss issue, not a demand-loss issue.\n\nEvidence: Clicks fell 42%, impressions only fell 8%, and average position dropped from 4.1 to 8.9. CTR also fell, which is expected when a page moves lower in search results.\n\nPriority: High.\n\nFix: Compare affected queries, inspect canonicals/indexability, review title/meta changes, check whether competitors gained stronger SERP features, and refresh the page around the

In [7]:
from huggingface_hub import login

login()

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "google/gemma-4-E2B-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.use_cache = False

print("Model loaded successfully:", MODEL_ID)

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Model loaded successfully: google/gemma-4-E2B-it


In [9]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

def formatting_func(example):
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

training_args = SFTConfig(
    output_dir="outputs/gemma4-e2b-seo-lora-v4-400",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=20,
    save_steps=100,
    report_to="none",
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    max_length=1024,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    peft_config=peft_config,
    formatting_func=formatting_func,
    processing_class=tokenizer,
)

trainer.train()

Applying formatting function to train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
20,2.942686
40,1.394462
60,1.169982
80,1.248475
100,1.208310
120,1.037011
140,1.134239
160,0.968939
180,0.982775
200,1.026491


TrainOutput(global_step=200, training_loss=1.3113370800018311, metrics={'train_runtime': 252.1236, 'train_samples_per_second': 1.587, 'train_steps_per_second': 0.793, 'total_flos': 1166097491814528.0, 'train_loss': 1.3113370800018311})

In [ ]:
!pip uninstall -y wandb

In [11]:
import re
import torch

def normalize_headings(text):
    text = text.strip()
    replacements = {
        r"Diagnosis\s*:": "Diagnosis:",
        r"Evidence\s*:": "Evidence:",
        r"Priority\s*:": "Priority:",
        r"Fix\s*:": "Fix:",
        r"Next action\s*:": "Next action:",
        r"Next action\s*\n": "Next action:\n",
    }
    for pattern, replacement in replacements.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text

def clean_first_complete_answer(text):
    text = normalize_headings(text)

    for marker in [
        "\nsystem\n",
        "\nmodel\n",
        "\ntype\n",
        "\nevidence\n",
        "\npriority\n",
        "\nfix\n",
        "\nnext action\n",
        "<start_of_turn>model",
        "<end_of_turn>",
    ]:
        if marker in text:
            text = text.split(marker)[0].strip()

    second_diag = text.find("\nDiagnosis:", 5)
    if second_diag != -1:
        text = text[:second_diag].strip()

    return normalize_headings(text)

def test_model_final_clean(user_prompt):
    test_messages = [
        {
            "role": "system",
            "content": """You are an evidence-led SEO and marketing assistant.

Answer using exactly these five headings:
Diagnosis:
Evidence:
Priority:
Fix:
Next action:

Rules:
- Include all five headings.
- Keep each section short.
- Do not add extra sections.
- Stop after Next action."""
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    test_prompt = tokenizer.apply_chat_template(
        test_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(test_prompt, return_tensors="pt").to(trainer.model.device)

    with torch.no_grad():
        outputs = trainer.model.generate(
            **inputs,
            max_new_tokens=350,
            do_sample=False,
            repetition_penalty=1.22,
            no_repeat_ngram_size=5,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    raw_response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    response = clean_first_complete_answer(raw_response)

    print(response)

    print("\n--- Heading check ---")
    for heading in ["Diagnosis:", "Evidence:", "Priority:", "Fix:", "Next action:"]:
        print(heading, "✅" if heading in response else "❌")

In [14]:
test_model_final_clean("Can we generate 5,000 location pages with the same copy?")

Diagnosis:
This is a low-quality scale risk that can harm local relevance if done without unique value for every page.

Evidence:
The prompt shows no proof of demand or uniqueness across thousands of locations; it also risks duplicate content penalties.

Priority:
High because this scales quickly from good to spammy unless validated by business need first.

Fix:
Create only high-demand service areas supported by real data before scaling beyond proven templates.

Next action:
Validate one template against actual search queries in your target area instead of generating hundreds immediately.

--- Heading check ---
Diagnosis: ✅
Evidence: ✅
Priority: ✅
Fix: ✅
Next action: ✅


In [13]:
trainer.model.eval()

try:
    trainer.model.gradient_checkpointing_disable()
except Exception as e:
    print("Could not disable gradient checkpointing:", e)

try:
    trainer.model.config.use_cache = True
except Exception as e:
    print("Could not set use_cache:", e)

print("Model set to inference mode")

Model set to inference mode


In [15]:
test_model_final_clean("The crawl says 20 important pages are noindex. What should we do first?")

Diagnosis:
This is a critical indexation issue that must be fixed before scaling to less important URLs or templates.

Evidence:
Crawl data shows many valuable pages have the noindex tag, which prevents them from ranking in search results even if they look good on page. Missing evidence includes affected URL value, GSC status, recent changes, and whether canonicals point correctly.

Priority:
Critical because it blocks organic visibility for potentially revenue-driving content. Fix this immediately with safe fixes like removing robots/noindex tags where appropriate.

Fix:
First confirm what changed (CMS update, plugin change). Then remove incorrect meta robots settings only on confirmed duplicate internal links while keeping correct ones intact. Follow up by regenerating sitemap and revalidating key landing pages.

Next action:
Export the list of indexed vs unindexed priority URLs and apply temporary fix rules based on business risk rather than fixing every single link at once.

--- He

In [16]:
test_model_final_clean("We don’t have GSC access. Diagnose the ranking drop anyway.")

Diagnosis:
The issue is that we cannot confirm whether rankings, impressions, or CTR changed without direct data.

Evidence:
Missing proof includes affected URLs, date range, page type mix, indexation status, crawl findings, PageSpeed metrics, recent site changes, and competitor movement.

Priority:
High if revenue pages dropped; Medium for informational support pages unless they lost clicks from important queries.

Fix:
Use GA4 organic landing/query reports as a proxy where available. Use SEMrush/Ahrefs to check query loss against competitors while waiting for GSC export permission. Then request GSC exports with specific filters (date, device, country).

Next action:
Export GA4 Organic Landing report by URL before asking for GSC permissions.

--- Heading check ---
Diagnosis: ✅
Evidence: ✅
Priority: ✅
Fix: ✅
Next action: ✅


In [17]:
test_model_final_clean("The PageSpeed score is low but conversions are up. Should we still fix it?")

Diagnosis:
PageSpeed should be treated as a performance risk, even if business metrics improve temporarily.

Evidence:
Low speed can affect mobile users who may convert later or return for repeat purchases; conversion lift must last across devices before fixing the page itself. Missing evidence includes affected URLs, device mix, Core Web Vitals, GSC data, GA4 events, and recent site changes.

Priority:
High until proven safe by real user impact rather than lab scores alone.

Fix:
Prioritize fixes that help both organic visibility (LCP/INP) and revenue paths while avoiding breaking existing functionality. Test one change at a time with controlled rollout.

Next action:
Segment current pages by device, query type, landing page, date range, and compare CWV against assisted conversions in GA4.

--- Heading check ---
Diagnosis: ✅
Evidence: ✅
Priority: ✅
Fix: ✅
Next action: ✅


In [18]:
trainer.model.save_pretrained("outputs/gemma4-e2b-seo-lora-v4-400")
tokenizer.save_pretrained("outputs/gemma4-e2b-seo-lora-v4-400")

print("Saved v4 400-example adapter")

Saved v4 400-example adapter


In [19]:
from huggingface_hub import login, create_repo

login()

repo_id = "RajeevSK25/gemma4-e2b-seo-lora-v4-400"

create_repo(repo_id, private=True, exist_ok=True, repo_type="model")

trainer.model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

print("Pushed v4 adapter:", repo_id)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed v4 adapter: RajeevSK25/gemma4-e2b-seo-lora-v4-400
